# PC-FNO3D réel (Grand Tunis) — OS7 : prévision du vent 3D à +24 et +48 h (Colab, A100)

Le **même modèle**, nourri par la **prévision GFS** au lieu d'ERA5 (cahier v12, section 18.10) :

1. prévisions GFS archivées (NOAA, AWS, depuis 2021) aux instants du cache, à +24 h et +48 h :
   seuls 38 champs par échéance (~34 Mo), découpés aussitôt sur les mailles du Grand Tunis
   (`pcfno3d.gfs`) ; reprise par instant sur le Drive (`os7/`) ;
2. **référence** : le modèle livré nourri par GFS, **sans affinage** ;
3. **affinage v5** : GFS +24 h en entrée, la même vérité, le même découpage, aucune station en entrée ;
4. **PREV1** (+24 h) et **PREV2** (+48 h) aux stations de l'INM sur le test 2022, contre GFS brut et
   la persistance ; **PREV3** : décision de traitement de la veille contre la décision mesurée ;
5. le champ 3D prévu contre la vérité générée (DSO1 à DSO5 du test) ;
6. **AGRI1 / AGRI2** du modèle livré (décisions à station cachée contre les stations).

Le test 2022 n'est lu qu'une fois par étape ; une étape déjà faite est sautée. La session s'arrête à
la fin. Limite : la qualité dépend de celle de GFS (le modèle n'avance pas l'atmosphère lui-même).

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

# Herbie : telechargement partiel des fichiers GFS (index .idx) ; cfgrib / eccodes : lecture GRIB2
!pip install -q herbie-data cfgrib eccodes 2>&1 | tail -1

## 2. Cache (3 parties) et observations

In [ ]:
# cache : les 3 parties du Drive -> disque local ; index fusionne
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (GT / f"cache_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())

# la grille complete (aneo_grid_points.csv) est dans jeu.zip
if not (GT / "best_dso_v7.pt").exists():
    raise FileNotFoundError("deposer best_dso_v7.pt (poids v7) dans MyDrive/pcfno3d_grand_tunis/")
lancer("controle", "python -u -m pcfno3d -c grand_tunis_v2_long_colab check")

# observations reelles du domaine (affinage et jugement)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print(sorted(p.name for p in OBS.iterdir()))

## 3. Mémoire du GPU

In [ ]:
# memoire : un pas complet (avant + arriere) sur la grille entiere ; recalcul des activations
# couche par couche si le GPU ne suffit pas (memes resultats, ~30 % plus lent)
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/grand_tunis_v5_gfs_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. GFS, référence, affinage v5, PREV1-3, test 3D, AGRI1-2, fin de la session

In [ ]:
from google.colab import runtime
import traceback
RUNS = GT / "runs"
OS7 = GT / "os7"
OS7.mkdir(exist_ok=True)
DECISION = RUNS / "decision_v4.json"
livre = json.loads(DECISION.read_text())["modele_livre"] if DECISION.exists() else "grand_tunis_v2_long"
CONF_LIVRE = {"grand_tunis_v4_dso": "grand_tunis_v4_dso_colab",
              "grand_tunis_v2_long": "grand_tunis_v2_long_colab"}[livre]
V5 = "grand_tunis_v5_gfs_colab"
R5 = RUNS / "grand_tunis_v5_gfs"
DONNEES = DON / "grand_tunis"
F = {e: OS7 / f"forcage_gfs_f{e}.csv" for e in (24, 48)}
print("modele livre :", livre)

def forcage(e):                    # le forcage GFS d'une echeance, copie la ou la configuration le lit
    return f"--set paths.forcage_era5={DONNEES / F[e].name}"

def copier_forcages():
    for e, f in F.items():
        for suffixe in (".csv", ".npz"):
            src = f.with_suffix(suffixe)
            if src.exists():
                shutil.copy2(src, DONNEES / src.name)

try:
    # 1. previsions GFS archivees : reprise par instant sur le Drive (quelques Ko chacun)
    for e in (24, 48):
        if not F[e].exists():
            lancer(f"gfs_f{e}", f"python -u -m pcfno3d.gfs --config {CONF_LIVRE} --echeance {e} "
                                f"--sortie {F[e]} --reprise {OS7 / f'gfs_f{e}_instants'} --travailleurs 8")
    copier_forcages()

    # 2. reference : le modele LIVRE nourri par GFS, sans affinage (meme entree que la v5)
    INIT = RUNS / livre / "best.pt"
    for e in (24, 48):
        d = OS7 / f"sans_affinage_f{e}"
        if not (d / "prevision_bilan.json").exists():
            try:
                lancer(f"prevision_sans_affinage_f{e}",
                       f"python -u scripts/evaluer_prevision.py -c {V5} {OPTS} {forcage(e)} --checkpoint {INIT} "
                       f"--obs {OBS} --echeance {e} --sortie {d}")
            except RuntimeError as exc:                 # code 3 = PREV non conforme : resultat ecrit
                print(exc)

    # 3. affinage v5 (GFS +24 h en entree, meme verite) depuis le modele livre
    h = R5 / "history.json"
    if not (h.exists() and any(r.get("epoch") == 11 for r in json.loads(h.read_text()))):
        lancer("entrainer_v5", f"python -u -m pcfno3d -c {V5} {OPTS} {forcage(24)} --set train.init_from={INIT} train")

    # 4. PREV1 (+24 h), PREV2 (+48 h) avec la v5 ; PREV3 : decisions de la veille
    for e in (24, 48):
        d = OS7 / f"v5_f{e}"
        if not (d / "prevision_bilan.json").exists():
            try:
                lancer(f"prevision_v5_f{e}", f"python -u scripts/evaluer_prevision.py -c {V5} {OPTS} {forcage(e)} "
                                             f"--obs {OBS} --echeance {e} --sortie {d}")
            except RuntimeError as exc:
                print(exc)
        if (d / "prevision_paires.csv").exists() and not (d / "decisions" / "decisions_agricoles.json").exists():
            lancer(f"prev3_f{e}", f"python -u scripts/verifier_decisions.py --paires {d / 'prevision_paires.csv'} "
                                  f"--sources prevision,gfs,persistance --sortie {d / 'decisions'}")

    # 5. le champ 3D prevu contre la verite generee (test 2022), a +24 h
    if not (R5 / "eval_pcfno3d" / "best_wcont" / "test_summary.json").exists():
        lancer("evaluer_v5_f24", f"python -u -m pcfno3d -c {V5} {OPTS} {forcage(24)} evaluate")

    # 6. AGRI1 / AGRI2 du modele livre : decisions a station cachee (validation reelle) contre les stations
    paires = RUNS / livre / "validation_reelle" / "test" / "stations_paires.csv"
    if paires.exists() and not (RUNS / livre / "decisions_agricoles" / "decisions_agricoles.json").exists():
        lancer("agri1_agri2", f"python -u scripts/verifier_decisions.py --paires {paires} "
                              f"--sources modele,era5,generateur --sortie {RUNS / livre / 'decisions_agricoles'}")
    for f in sorted(OS7.glob("*/prevision_bilan.json")):
        print(f.parent.name, json.dumps(json.loads(f.read_text())["verdicts"].get("toutes", {}), ensure_ascii=False))
    # bilan de conformite au cahier v12 (lecture seule) : runs/bilan_conformite.md
    lancer("bilan_conformite", f"python -u scripts/bilan_conformite.py --runs {RUNS}")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()